# 面试问题：长时 Agent 怎样压缩 Context，同时保留约束、任务状态、来源并抵御记忆投毒？

        ## 可直接复述的回答主线

        1. Context Compaction 不是把最近消息截断，而是把事实、约束、决策和任务转换成带来源的结构化记忆。
2. 最近 N 条基线容易丢掉早期但仍生效的区域、配额和审批约束。
3. 结构化压缩应按 key 合并状态，同时保留 turn、source、authority 和任务完成状态。
4. 发生冲突时不能只看时间，新近低权威检索内容不应覆盖用户明确约束或权威工具回读。
5. 结果应比较预算、关键事实覆盖率、冲突选择和可追溯 provenance。
6. 生产系统还需要 schema 版本、删除权、敏感信息治理、摘要评测和回放审计。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是一段部署 Agent 的十条脱敏事件流，包含用户约束、工具事实、Agent 任务、发布决策和一条低权威检索投毒。目标是在有限上下文预算中保留 region、max_replicas、approval_required、queue_depth 和 rollout 五个关键字段。

In [1]:
events = [{"turn": 1, "source": "user", "kind": "constraint", "key": "region", "value": "cn-north", "status": "active"}, {"turn": 2, "source": "tool", "kind": "fact", "key": "deployment_version", "value": "v17", "status": "observed"}, {"turn": 3, "source": "user", "kind": "constraint", "key": "max_replicas", "value": 4, "status": "active"}, {"turn": 4, "source": "agent", "kind": "task", "key": "investigate_latency", "value": "检查GPU队列", "status": "pending"}, {"turn": 5, "source": "tool", "kind": "fact", "key": "p95_ms", "value": 3200, "status": "observed"}, {"turn": 6, "source": "retrieved", "kind": "fact", "key": "region", "value": "us-west", "status": "untrusted"}, {"turn": 7, "source": "user", "kind": "decision", "key": "rollout", "value": "canary-10%", "status": "approved"}, {"turn": 8, "source": "tool", "kind": "fact", "key": "queue_depth", "value": 42, "status": "observed"}, {"turn": 9, "source": "agent", "kind": "task", "key": "investigate_latency", "value": "检查GPU队列", "status": "done"}, {"turn": 10, "source": "user", "kind": "constraint", "key": "approval_required", "value": True, "status": "active"}]  # 定义十条有来源、类型和状态的 Agent 事件。
required_keys = {"region", "max_replicas", "approval_required", "queue_depth", "rollout"}  # 定义下一轮部署决策必须保留的五个字段。
authority = {"retrieved": 1, "agent": 2, "tool": 3, "user": 4}  # 定义冲突解决使用的来源权威等级。
print("教学实验输入：部署 Agent 事件流")  # 标记下表是脱敏离线事件。
print("turn source     kind        key                  value        status")  # 输出事件字段表头。
for event in events:  # 逐条展示十轮结构化事件。
    print(f"{event['turn']:>4} {event['source']:<10} {event['kind']:<11} {event['key']:<20} {str(event['value']):<12} {event['status']}")  # 输出当前事件的来源和状态。

教学实验输入：部署 Agent 事件流
turn source     kind        key                  value        status
   1 user       constraint  region               cn-north     active
   2 tool       fact        deployment_version   v17          observed
   3 user       constraint  max_replicas         4            active
   4 agent      task        investigate_latency  检查GPU队列      pending
   5 tool       fact        p95_ms               3200         observed
   6 retrieved  fact        region               us-west      untrusted
   7 user       decision    rollout              canary-10%   approved
   8 tool       fact        queue_depth          42           observed
   9 agent      task        investigate_latency  检查GPU队列      done
  10 user       constraint  approval_required    True         active


## 2. Baseline / 基线：只保留最近四条事件

最近 N 条实现简单，但当前窗口恰好丢掉 turn 1 的 region 和 turn 3 的 max_replicas。它保留了一条任务完成记录，却没有保留任务为何产生和早期硬约束。

In [2]:
recent_window = events[-4:]  # 取最近四条事件模拟基于窗口的朴素压缩。
recent_keys = {event["key"] for event in recent_window}  # 提取窗口仍能回答的字段。
baseline_coverage = len(required_keys & recent_keys) / len(required_keys)  # 计算关键决策字段覆盖率。
print("Baseline 最近四条上下文")  # 标记当前输出属于时间窗口截断。
for event in recent_window:  # 逐条展示截断后实际留存内容。
    print(f"turn={event['turn']} {event['source']} {event['key']}={event['value']} status={event['status']}")  # 输出当前窗口事件。
print(f"关键字段覆盖={sorted(required_keys & recent_keys)}，coverage={baseline_coverage:.1%}，丢失={sorted(required_keys - recent_keys)}")  # 直接展示截断损失。

Baseline 最近四条上下文
turn=7 user rollout=canary-10% status=approved
turn=8 tool queue_depth=42 status=observed
turn=9 agent investigate_latency=检查GPU队列 status=done
turn=10 user approval_required=True status=active
关键字段覆盖=['approval_required', 'queue_depth', 'rollout']，coverage=60.0%，丢失=['max_replicas', 'region']


## 3. 底层实现：带 authority 和 provenance 的结构化合并

对 constraint、fact、decision 按 key 选保留项：更高 authority 优先，同 authority 才使用更新 turn。task 单独保留最新状态，所有输出携带原始 turn 和 source。

In [3]:
def compact_memory(event_stream):  # 把事件流压缩成具有来源的结构化记忆。
    state = {}  # 保存事实、约束和决策的当前权威版本。
    tasks = {}  # 单独保存每个任务的最新生命周期状态。
    decisions = []  # 保存冲突选择过程供审计。
    for event in event_stream:  # 按时间顺序处理全部事件。
        if event["kind"] == "task":  # 任务需要保留最新 pending 或 done 状态。
            previous = tasks.get(event["key"])  # 读取当前任务已有状态。
            if previous is None or event["turn"] > previous["turn"]:  # 只用更晚事件推进任务生命周期。
                tasks[event["key"]] = event  # 提交当前任务最新状态。
            continue  # 任务不进入普通 key 冲突逻辑。
        previous = state.get(event["key"])  # 读取当前 key 已选择的事件。
        if previous is None:  # 首次出现的 key 可以直接保留。
            state[event["key"]] = event  # 写入首个有来源值。
            decisions.append((event["key"], "首次写入", event["source"], event["value"]))  # 记录首次选择原因。
            continue  # 进入下一条事件。
        previous_rank = authority[previous["source"]]  # 读取已保存来源权威等级。
        current_rank = authority[event["source"]]  # 读取新事件来源权威等级。
        should_replace = current_rank > previous_rank or (current_rank == previous_rank and event["turn"] > previous["turn"])  # 更高权威优先，同权威才按新旧选择。
        reason = "更高权威/同权威更新" if should_replace else "拒绝低权威覆盖"  # 生成可解释冲突决策。
        decisions.append((event["key"], reason, event["source"], event["value"]))  # 保存当前冲突审计记录。
        if should_replace:  # 只有满足权威规则时才替换记忆。
            state[event["key"]] = event  # 提交新的权威版本。
    return state, tasks, decisions  # 返回结构化状态、任务和冲突账本。
memory_state, memory_tasks, merge_decisions = compact_memory(events)  # 对十轮事件执行结构化压缩。
print("结构化记忆状态")  # 标记下表是压缩后可注入上下文的事实。
print("key                  value        source   turn  status")  # 输出记忆表头。
for key in sorted(memory_state):  # 按 key 稳定展示最终权威状态。
    event = memory_state[key]  # 读取当前 key 的保留事件。
    print(f"{key:<20} {str(event['value']):<12} {event['source']:<8} {event['turn']:>4}  {event['status']}")  # 输出值及 provenance。
print("任务状态：", {key: event["status"] for key, event in memory_tasks.items()})  # 展示任务完成状态没有混入事实冲突。

结构化记忆状态
key                  value        source   turn  status
approval_required    True         user       10  active
deployment_version   v17          tool        2  observed
max_replicas         4            user        3  active
p95_ms               3200         tool        5  observed
queue_depth          42           tool        8  observed
region               cn-north     user        1  active
rollout              canary-10%   user        7  approved
任务状态： {'investigate_latency': 'done'}


## 4. 结果表与结果解读

在相同“只输出当前状态”的目标下，结构化记忆恢复五个关键字段，并保留来源。它不是保留所有原文，而是保留下一步决策真正需要的最小状态。

In [4]:
structured_keys = set(memory_state)  # 提取结构化状态能够回答的字段。
structured_coverage = len(required_keys & structured_keys) / len(required_keys)  # 计算结构化记忆关键字段覆盖率。
print("压缩策略              保留事件/状态数  关键字段覆盖  region值   region来源")  # 输出两种压缩策略对照表头。
baseline_region = next((event for event in recent_window if event["key"] == "region"), None)  # 查找最近窗口是否仍含 region。
print(f"{'最近4条':<21} {len(recent_window):>13} {baseline_coverage:>12.1%} {str(None if baseline_region is None else baseline_region['value']):>10} {'无' if baseline_region is None else baseline_region['source']:>12}")  # 输出时间窗口基线结果。
print(f"{'结构化权威记忆':<19} {len(memory_state) + len(memory_tasks):>13} {structured_coverage:>12.1%} {str(memory_state['region']['value']):>10} {memory_state['region']['source']:>12}")  # 输出结构化压缩结果。
print("结果解读：结构化方案恢复早期仍生效约束，并能说明 region 来自 turn 1 用户，而不是只给出一个无来源摘要。")  # 解释覆盖率和 provenance 的意义。

压缩策略              保留事件/状态数  关键字段覆盖  region值   region来源
最近4条                              4        60.0%       None            无
结构化权威记忆                         8       100.0%   cn-north         user
结果解读：结构化方案恢复早期仍生效约束，并能说明 region 来自 turn 1 用户，而不是只给出一个无来源摘要。


## 5. 失败案例与修正

如果同 key 永远让最新事件覆盖，turn 6 的低权威检索文本会把 region 改为 us-west。权威优先规则拒绝该覆盖，并在冲突账本中留下原因。

In [5]:
naive_latest = {}  # 创建只按时间覆盖的错误记忆。
for event in events:  # 按到达顺序盲目更新同名 key。
    naive_latest[event["key"]] = event  # 让任何新来源覆盖已有权威值。
poisoned_region = naive_latest["region"]  # 读取被低权威检索覆盖后的 region。
fixed_region = memory_state["region"]  # 读取权威合并保留的 region。
region_decisions = [decision for decision in merge_decisions if decision[0] == "region"]  # 提取 region 的冲突审计记录。
print(f"错误行为：latest-wins region={poisoned_region['value']} source={poisoned_region['source']} turn={poisoned_region['turn']}")  # 展示记忆投毒覆盖结果。
print(f"修正行为：authority-first region={fixed_region['value']} source={fixed_region['source']} turn={fixed_region['turn']}")  # 展示权威来源被保留。
print("冲突账本：", region_decisions)  # 输出拒绝低权威覆盖的审计原因。

错误行为：latest-wins region=us-west source=retrieved turn=6
修正行为：authority-first region=cn-north source=user turn=1
冲突账本： [('region', '首次写入', 'user', 'cn-north'), ('region', '拒绝低权威覆盖', 'retrieved', 'us-west')]


## 6. 生产边界

实际 Context Compaction 还需处理自然语言实体对齐、schema 迁移、撤回/删除、PII、跨会话权限、摘要模型幻觉和 token 精确计数。权威表应由产品安全策略配置而非硬编码。

In [6]:
serialized_memory = [f"{key}={event['value']}|src={event['source']}|turn={event['turn']}" for key, event in sorted(memory_state.items())]  # 构造可注入下一轮 prompt 的紧凑有来源记录。
approximate_tokens = sum(max(1, len(item) // 2) for item in serialized_memory)  # 用字符近似估算教学上下文 Token 成本。
print("紧凑记忆片段：", serialized_memory)  # 展示真正会交给下一轮 Agent 的结构。
print(f"教学近似Token={approximate_tokens}，生产中需替换为实际 tokenizer 并设置硬预算。")  # 明确预算估算的替换点。

紧凑记忆片段： ['approval_required=True|src=user|turn=10', 'deployment_version=v17|src=tool|turn=2', 'max_replicas=4|src=user|turn=3', 'p95_ms=3200|src=tool|turn=5', 'queue_depth=42|src=tool|turn=8', 'region=cn-north|src=user|turn=1', 'rollout=canary-10%|src=user|turn=7']
教学近似Token=113，生产中需替换为实际 tokenizer 并设置硬预算。


## 7. 最小回归测试

只验证事件规模、覆盖率、任务终态、来源和投毒修正。

In [7]:
assert len(events) >= 5  # 保证案例包含足够长的真实字段事件流。
assert structured_coverage > baseline_coverage  # 保证结构化压缩比最近窗口保留更多关键状态。
assert memory_tasks["investigate_latency"]["status"] == "done"  # 保证任务生命周期推进到最新终态。
assert fixed_region["value"] == "cn-north" and fixed_region["source"] == "user"  # 保证低权威检索不能覆盖用户区域约束。
assert poisoned_region["value"] != fixed_region["value"]  # 保证失败案例真实展示 latest-wins 投毒。